# 03 · Train the motion model (J, Z)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsemaniJoshua/SignLens/blob/main/notebooks/03_train_motion_model.ipynb)

J and Z are defined by **movement**, so one photo can't identify them. This model looks at about 1.5 seconds of hand landmarks: the changing handshape **and the path the wrist travels**. The old model lost that path, because every frame used absolute screen positions mixed with body pose.

How the app uses it: the static model answers by default; when the hand is actually moving, the motion model takes over (`signlens_core/inference.py`).

**Important about data:** the repo has only **1 J clip and 2 Z clips**. The notebook runs and exports a working model with heavy augmentation, but a reliable motion model needs **about 20+ clips per sign from several people**. Record them on a phone (3 seconds, one J or Z per clip) into `MyDrive/SignLens/my_videos/J/` and `.../Z/`, then re-run notebook 01 and this one.

The model also learns an **`other`** class from generated clips (the hand moving between ordinary letter shapes), so moving into position for the next letter isn't mistaken for J or Z.

In [ ]:
#@title Setup: get the code and data (run this first)
import os, sys, subprocess

REPO_URL = "https://github.com/AsemaniJoshua/SignLens"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    """Run a shell command; stop the notebook if it fails."""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    REPO_DIR = "/content/SignLens"
    if not os.path.isdir(REPO_DIR):
        # Shallow + sparse clone: only notebooks/ and the small keypoint files
        # (~225 MB, a few seconds). Skips node_modules, videos and images.
        sh(f"GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch {BRANCH} "
           f"--filter=blob:none --sparse {REPO_URL} {REPO_DIR}")
        sh(f"cd {REPO_DIR} && git sparse-checkout set --no-cone /notebooks/ "
           "/model/dataset/static_keypoints_optimized/ "
           "'/model/dataset/video_keypoints_no_face/*/[0-9].npy' "
           "'/model/initial build/dataset/' /model/model_endpoints/app/")
    sh("pip install -q mediapipe")
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/SignLens"   # outputs survive the session
else:
    REPO_DIR = os.environ.get("SIGNLENS_REPO_DIR", os.path.abspath(".."))
    WORK_DIR = os.environ.get("SIGNLENS_WORK_DIR", os.path.join(REPO_DIR, "work"))

MODEL_DIR = os.path.join(REPO_DIR, "model")
DATA_DIR = os.path.join(WORK_DIR, "data")
OUT_DIR = os.path.join(WORK_DIR, "models")
for d in (WORK_DIR, DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))

import numpy as np
import signlens_core as sc
QUICK = os.environ.get("SIGNLENS_QUICK") == "1"   # tiny settings for automated tests
print(f"signlens_core {sc.__version__} | repo data: {MODEL_DIR} | outputs: {WORK_DIR}")

In [ ]:
#@title Options
EPOCHS = 60  #@param {type:"integer"}
BATCH = 64  #@param {type:"integer"}
OTHER_PER_EPOCH = 400  #@param {type:"integer"}
SEED = 42
if QUICK:
    EPOCHS = 3

## 1. Load clips, add the reject class

In [ ]:
from collections import Counter
from signlens_core.sequences import synthetic_other_motion

seq_path = os.path.join(DATA_DIR, "motion_sequences.npz")
if os.path.exists(seq_path):
    samples, seq_labels, seq_groups, seq_sources = sc.load_sequences(seq_path)
else:
    print("motion_sequences.npz not found -> using repo videos only (run notebook 01 for more)")
    rows = [(s, l, g) for s, l, g in sc.load_repo_video_sequences(MODEL_DIR) if l in ("J", "Z")]
    samples, seq_labels, seq_groups = map(list, zip(*rows)); seq_sources = ["repo_video"] * len(rows)
seq_labels, seq_groups = np.asarray(seq_labels, dtype=object), np.asarray(seq_groups, dtype=object)
counts = Counter(seq_labels)
print("real clips per sign:", dict(counts))
for lab, n in counts.items():
    if n < 20:
        print(f"  !! {lab}: only {n} clip(s). The model will run, but record ~20+ from several people for real accuracy.")

# handshapes for the generated 'other' clips come from the static dataset
static_path = os.path.join(DATA_DIR, "static_landmarks.npz")
static = sc.StaticSet.load(static_path) if os.path.exists(static_path) else sc.load_repo_youtube_frames(MODEL_DIR)
P_static = static.normalized()
rng = np.random.default_rng(SEED)
val_other = synthetic_other_motion(P_static, np.random.default_rng(SEED + 1), 200)

LABELS = sc.sort_labels(counts) + ["other"]
lab2i = {l: i for i, l in enumerate(LABELS)}
print("model classes:", LABELS)

## 2. Training data stream

Every epoch: each real clip appears many times with fresh augmentation (speed changes, angle, hand size, mirror), plus a fresh batch of generated `other` clips.

In [ ]:
def epoch_data(real_idx, rng, reps=None):
    """One epoch's (X, y): augmented real clips + new 'other' clips."""
    real_idx = np.asarray(real_idx)
    reps = reps or max(1, OTHER_PER_EPOCH // max(1, len(real_idx)) // max(1, len(counts)))
    order = np.repeat(real_idx, reps)
    X_real = sc.batches.sequence_batch(samples, order, rng, augment=True) if len(order) else np.zeros((0, sc.SEQ_LEN, sc.SEQ_FEATURE_DIM))
    others = synthetic_other_motion(P_static, rng, OTHER_PER_EPOCH)
    X_other = sc.batches.sequence_batch(others, range(len(others)), rng, augment=True)
    X = np.concatenate([X_real, X_other]).astype(np.float32)
    Y = np.concatenate([[lab2i[seq_labels[i]] for i in order], np.full(len(others), lab2i["other"])]).astype(int)
    perm = rng.permutation(len(X))
    return X[perm], Y[perm]

X_demo, Y_demo = epoch_data(np.arange(len(samples)), rng)
print("one epoch:", X_demo.shape, dict(Counter(np.array(LABELS)[Y_demo])))

## 3. Model and training

In [ ]:
import keras
from keras import layers
keras.utils.set_random_seed(SEED)

def build_model(adapt_X):
    norm = layers.Normalization(axis=-1, name="feature_scaling")
    norm.adapt(adapt_X)
    inp = keras.Input((sc.SEQ_LEN, sc.SEQ_FEATURE_DIM), name="sequence_features")
    x = norm(inp)
    for filters, stride in ((64, 1), (96, 2), (128, 1)):
        x = layers.Conv1D(filters, 5, strides=stride, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("gelu")(x)
    x = layers.Concatenate()([layers.GlobalAveragePooling1D()(x), layers.GlobalMaxPooling1D()(x)])
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="gelu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(len(LABELS), activation="softmax", name="probs")(x)
    m = keras.Model(inp, out, name="signlens_motion")
    m.compile(optimizer=keras.optimizers.AdamW(1e-3, weight_decay=1e-4),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def train(real_idx, epochs=EPOCHS, verbose=0):
    r = np.random.default_rng(SEED)
    m = build_model(epoch_data(real_idx, r)[0])
    for ep in range(epochs):
        X, Y = epoch_data(real_idx, r)          # fresh augmentation every epoch
        h = m.fit(X, Y, batch_size=BATCH, epochs=1, verbose=0)
        if verbose and (ep + 1) % 10 == 0:
            print(f"epoch {ep + 1}: loss {h.history['loss'][-1]:.3f} acc {h.history['accuracy'][-1]:.3f}")
    return m

def predict_labels(m, X):
    return np.array(LABELS)[m.predict(X, verbose=0).argmax(1)]

train_and_predict = lambda tr_idx, X: predict_labels(train(tr_idx), X)

## 4. Evaluation: leave one recording out

Each real clip is held out in turn: train on the rest, then ask the model about that clip. It only runs for signs with at least 2 clips, and the result only means something once you have recorded more. The generated `other` clips are also checked, to confirm ordinary movement is rejected.

In [ ]:
testable = [i for i in range(len(samples)) if counts[seq_labels[i]] >= 2]
true_l, pred_l = [], []
for i in testable:
    rest = np.array([j for j in range(len(samples)) if j != i])
    p = train_and_predict(rest, sc.batches.sequence_batch(samples, [i]))[0]
    true_l.append(seq_labels[i]); pred_l.append(p)
    print(f"  held out {seq_groups[i]:35s} true {seq_labels[i]}  predicted {p}")
if true_l:
    sc.evaluate.report(true_l, pred_l, LABELS, title="leave-one-recording-out (real clips)")
else:
    print("No sign has 2+ clips yet, so no honest evaluation is possible. Record more clips.")

final = train(np.arange(len(samples)), verbose=1)
other_pred = predict_labels(final, sc.batches.sequence_batch(val_other, range(len(val_other))))
print(f"\n'other' movement correctly rejected: {np.mean(other_pred == 'other'):.1%} of {len(val_other)} unseen generated clips")

## 5. Export

In [ ]:
import tensorflow as tf
from signlens_core import inference
keras_path = os.path.join(OUT_DIR, "motion_model.keras")
tflite_path = os.path.join(OUT_DIR, "motion_model.tflite")
labels_path = os.path.join(OUT_DIR, "motion_labels.json")
final.save(keras_path)

@tf.function(input_signature=[tf.TensorSpec([1, sc.SEQ_LEN, sc.SEQ_FEATURE_DIM], tf.float32)])
def serve(x):
    return final(x, training=False)
def to_tflite(concrete_fn, model):
    """Convert with standard TFLite ops; fall back to TF ops if something isn't supported."""
    conv = tf.lite.TFLiteConverter.from_concrete_functions([concrete_fn], model)
    try:
        return conv.convert()
    except Exception as e:
        print("standard TFLite ops failed (", str(e)[:120], ") -> retrying with TF ops (needs the Flex delegate on mobile)")
        conv = tf.lite.TFLiteConverter.from_concrete_functions([concrete_fn], model)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
        return conv.convert()
open(tflite_path, "wb").write(to_tflite(serve.get_concrete_function(), final))
sc.save_labels(labels_path, LABELS, {"model": "motion", "seq_len": sc.SEQ_LEN,
                                     "feature_dim": sc.SEQ_FEATURE_DIM, "signlens_core": sc.__version__,
                                     "real_clips_per_class": dict(counts), "reject_label": "other",
                                     "motion_threshold_palms": 1.0, "window_seconds": 1.5})
check = sc.batches.sequence_batch(samples, range(min(len(samples), 20)))
diff = np.abs(inference.load_predict_fn(tflite_path)(check) - final.predict(check, verbose=0)).max()
print(f"TFLite vs Keras max difference: {diff:.2e}", "OK" if diff < 1e-3 else "!! CHECK EXPORT")
for p_ in (keras_path, tflite_path, labels_path):
    print(f"saved {p_}  ({os.path.getsize(p_) / 1024:.0f} KB)")

### Using both models together

```python
from signlens_core.inference import SignPredictor
predictor = SignPredictor("static_letters_model.tflite", "static_letters_labels.json",
                          motion_model="motion_model.tflite", motion_labels="motion_labels.json")
# for every camera frame:
result = predictor.update(landmarks, handedness, t_seconds, aspect=w / h)
```

When the motion model says `other`, the static model's answer is used. See `notebooks/README.md` for the full API example.